# Optical responsivity fitting
This notebook shows how to fit the fractional frequency shift of a KID versus absorbed (optical or loading) power with `citkid.responsivity`, using synthetic data.

For a KID whose quasiparticle density is set by absorbed power and recombination, the frequency responsivity rolls off as

$$\frac{dx}{dP} = R_0 \left(1 + \frac{P}{P_0}\right)^{-1/2},$$

where $R_0$ is the responsivity at $P = 0$ and $P_0$ is the power at which it has rolled off by $\sqrt{2}$. The fitters fit its integral, $x(P)$. With $x = (f_r - f_1) / f_1$ measured relative to an assumed zero-power frequency $f_1$ (e.g. $f_r$ at the lowest power), `citkid.responsivity.funcs` has two integrated forms:

| function | model | extra parameter |
|---|---|---|
| `responsivity_int_x0(P, R_0, P_0, x_0)` | $x = 2R_0P_0\left(\sqrt{1 + P/P_0} - 1\right) + x_0$ | $x_0$: $x$ at $P = 0$, so $f_0 = f_1 (1 + x_0)$ |
| `responsivity_int(P, R_0, P_0, c)` | $x = c \cdot 2R_0P_0\left(\sqrt{1 + P/P_0} - 1\right) + c - 1$ | $c = f_0 / f_1$ |

In both, $R_0 < 0$ for a frequency that falls with power, as it does for a KID. Both are compiled with numba: `P` must be a float64 array.

Sections:
1. Synthetic $x(P)$ data
2. Fitting with `fit_responsivity_int_x0`
3. Fitting with `fit_responsivity_int`
4. Fitting with $R_0$ fixed
5. The same fit in the sweep-series pipeline (`citkid.pipeline.sweep_series.responsivity_fit`)

`citkid.responsivity.funcs.responsivity(P, R_0, P_0)` evaluates $dx/dP$ itself. It is the derivative of both integrated forms (with $c = 1$), so a fitted $R_0$ can be passed to it directly.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from citkid.responsivity.funcs import (responsivity, responsivity_int,
                                       responsivity_int_x0)
from citkid.responsivity.fitter import (fit_responsivity_int,
                                        fit_responsivity_int_x0,
                                        fit_responsivity_int_x0_fixed_R0)
from citkid.pipeline.sweep_series import responsivity_fit

def print_fit(names, p_true, popt, perr):
    """Print injected and fitted parameters, and the pull (fit - true) / err."""
    print(f"{'':>4} {'true':>14} {'fit':>14} {'err':>10} {'pull':>6}")
    for name, t, p, e in zip(names, p_true, popt, perr):
        pull = (p - t) / e if e > 0 else np.nan
        print(f'{name:>4} {t:14.9g} {p:14.9g} {e:10.3g} {pull:6.2f}')

## 1. Synthetic data
We simulate a resonator with $f_0 = 500$ MHz, $R_0 = -5\times10^8$ W$^{-1}$ and $P_0 = 10$ fW, measured at 20 powers from 0.1 fW to 10 pW, with 20 Hz of Gaussian noise on $f_r$ (fixed seed). As in a real measurement, $f_0$ is unknown, so $x$ is computed relative to $f_1$, the measured $f_r$ at the lowest power.

The fitters work best with at least three points at $P \gg P_0$ (for the guess of $P_0$) and the lowest power at $P \ll P_0$.

In [ ]:
rng = np.random.default_rng(1)
f0 = 500e6
R0 = -5e8 # 1 / W
P0 = 1e-14 # W

P = np.geomspace(1e-16, 1e-11, 20)
fr_err = 20. # Hz
fr = f0 * (1 + responsivity_int_x0(P, R0, P0, 0.))
fr = fr + fr_err * rng.normal(size = P.size)

f1 = fr[0]
x = (fr - f1) / f1
x_err = np.full(P.size, fr_err / f1)
x0 = f0 / f1 - 1 # true x at P = 0 in this calibration
print(f'f1 - f0 = {f1 - f0:.1f} Hz, so x0 = {x0:.3g}')

fig, axs = plt.subplots(1, 2, figsize = [10, 4], layout = 'tight')
axs[0].errorbar(P, x * 1e6, x_err * 1e6, fmt = '.')
axs[0].set(xscale = 'log', xlabel = 'Power (W)', ylabel = r'$x$ (ppm)')
# Responsivity of the true model, R_0 (1 + P / P0)^(-1/2) (negative)
P_model = np.geomspace(P.min(), P.max(), 200)
axs[1].plot(P_model, np.abs(responsivity(P_model, R0, P0)))
axs[1].axvline(P0, color = '0.5', ls = ':', label = r'$P_0$')
axs[1].set(xscale = 'log', yscale = 'log', xlabel = 'Power (W)',
           ylabel = r'$|dx/dP|$ (1/W)')
axs[1].legend()
plt.show()

## 2. Fitting with `fit_responsivity_int_x0`
`fit_responsivity_int_x0(power, x, f1, x_err = ...)` fits `[R0, P0, x0]` and returns

- `p0`: the guess (`citkid.responsivity.guess.guess_p0_responsivity_int_x0`),
- `popt`, `perr`: the fit parameters and their errors, in 1/W, W and Hz/Hz (NaN if the fit failed),
- `f0`, `f0err`: the zero-power frequency $f_1(1 + x_0)$ and its error.

Pass `x_err` if you know the errors on $x$. Without it, points are weighted equally and the parameter errors are scaled by the fit residuals. `guess = [R0, P0, x0]` (unscaled units) replaces the automatic guess.

In [ ]:
p0, popt, perr, f0_fit, f0_err = fit_responsivity_int_x0(P, x, f1,
                                                         x_err = x_err)
print_fit(['R0', 'P0', 'x0'], [R0, P0, x0], popt, perr)
print(f'\nf0 = {f0_fit:.1f} +- {f0_err:.1f} Hz (true {f0:.1f} Hz)')

fig, ax = plt.subplots(figsize = [5, 4], layout = 'tight')
ax.errorbar(P, x * 1e6, x_err * 1e6, fmt = '.', label = 'data')
ax.plot(P_model, responsivity_int_x0(P_model, *popt) * 1e6, '--',
        label = 'fit')
ax.plot(P_model, responsivity_int_x0(P_model, *p0) * 1e6, ':k',
        label = 'guess')
ax.set(xscale = 'log', xlabel = 'Power (W)', ylabel = r'$x$ (ppm)')
ax.legend()
plt.show()

## 3. Fitting with `fit_responsivity_int`
`fit_responsivity_int(power, x, f1, x_err = ...)` fits the multiplicative form, `[R0, P0, c]`, and returns `f0 = c f1`. The two forms are equivalent to first order in $x$, so the results agree closely. (Unlike `fit_responsivity_int_x0`, its `guess` argument is in the fitter's scaled units, `[R0 / 1e9, P0 * 1e16, c]`.)

In [ ]:
p0, popt_c, perr_c, f0_fit_c, f0_err_c = fit_responsivity_int(P, x, f1,
                                                              x_err = x_err)
print_fit(['R0', 'P0', 'c'], [R0, P0, f0 / f1], popt_c, perr_c)
print(f'\nf0 = {f0_fit_c:.1f} +- {f0_err_c:.1f} Hz (true {f0:.1f} Hz)')

## 4. Fitting with $R_0$ fixed
When $P_0$ is above the measured powers, $R_0$ and $P_0$ are strongly correlated. If $R_0$ is known (e.g. from a model of the detector), `fit_responsivity_int_x0_fixed_R0(power, x, f1, R0, x_err = ...)` fits only `[P0, x0]`. It returns `popt = [R0, P0, x0]` with a zero error on $R_0$.

In [ ]:
p0, popt, perr, f0_fit, f0_err = fit_responsivity_int_x0_fixed_R0(
    P, x, f1, R0, x_err = x_err
)
print_fit(['R0', 'P0', 'x0'], [R0, P0, x0], popt, perr)

## 5. In the sweep-series pipeline
`citkid.pipeline.sweep_series.responsivity_fit()` wraps `fit_responsivity_int_x0` as a `SeriesXYFit` for `run_iq_series` (with `x_values` the powers in W and `y_param_name = 'fr'`). Its fit takes $f_1$ as $f_r$ at the lowest power, fits $x = (f_r - f_1) / f_1$ without errors, and returns `[resp_R0, resp_P0, resp_x0, resp_f1]`. Its model is $f_r = f_1(1 + x(P))$. The series window calls the same methods used here, `run` and `curve`.

In [ ]:
xy_fit = responsivity_fit()
popt_series, p0_series = xy_fit.run(P, fr)
print(xy_fit.describe(popt_series))

P_curve, fr_curve = xy_fit.curve(P, popt_series, log_x = True)
fig, ax = plt.subplots(figsize = [5, 4], layout = 'tight')
ax.plot(P, (fr - f0) / 1e3, '.', label = 'data')
ax.plot(P_curve, (fr_curve - f0) / 1e3, '--', label = 'SeriesXYFit model')
ax.set(xscale = 'log', xlabel = 'Power (W)',
       ylabel = rf'$f_r - {f0 / 1e6:.0f}$ MHz (kHz)')
ax.legend()
plt.show()